# NeoOLAF × RAGTree — Parallel 5-document validation

This notebook reruns the **same fixed five documents** for:

1. **EventStoryLine** — frozen `v1.7`
2. **FinCausal** — frozen `unified-v1.3.1-selection-hotfix`

The purpose is an **engineering parallelism/reproducibility check**, not new tuning and not a replacement for the already completed scientific smoke-5.

### Parallel settings

- **EventStoryLine:** `DOCUMENT_WORKERS = 5`, `LAYER_WORKERS = 4`
  - all 5 fixed documents are launched concurrently;
  - each document can use up to 4 native layer workers.
- **FinCausal:** `DOCUMENT_WORKERS = 5`, `LAYER_WORKERS = 1`
  - all 5 fixed documents are launched concurrently;
  - no extra 4-way layer parallelism is enabled for FinCausal.

The datasets are still sequential at dataset level: all EventStoryLine futures finish first, then the five FinCausal futures are launched.

### Safety

- Separate run root; original one-doc/smoke results are untouched.
- Development manifest is read-only.
- Gold is stripped before Layer 0.
- Gold is used only after Layer 12 for evaluation.
- Per-document run directories are isolated.
- Persistent progress allows resume without rerunning completed documents.
- Exact UTC start/end windows are saved for later OpenRouter usage/token inspection.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import os, sys, json, time, traceback, shutil, re
from pprint import pprint

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def find_project_root():
    candidates = []
    env = os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env:
        candidates.append(Path(env))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p / "src" / "neoolaf").is_dir() and (p / "examples").is_dir():
            return p.resolve()
    raise FileNotFoundError(
        "NeoOLAF project root not found. Set NEOOLAF_PROJECT_ROOT."
    )

PROJECT_ROOT = find_project_root()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"

for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate
import ragtree_dataset_adapters_v1_8 as adapters
import eventstoryline_native_ablation_v1_7 as esl_v17

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Adapter self-test:")
pprint(adapters.offline_self_test())


PROJECT_ROOT: C:\Users\galencarmedeiro\NeoOLAF
Adapter self-test:
{'cache_root': 'C:\\Users\\GALENC~1\\AppData\\Local\\Temp\\neoolaf_ragtree_unified4_v1_1',
 'causalbank': 'deterministic all ordered non-self pairs; no pair-level LLM '
               'pruning',
 'causalbank_hash_a': 'EVENT_0cc175b9c0f1b6a8',
 'causalbank_relations': ['BECAUSE', 'THEREFORE'],
 'datasets': ['fincausal', 'maven_ere', 'causalbank'],
 'fincausal': 'deterministic boundary union + role-hint pair fallback',
 'maven_ere': 'frozen/reused v1.4 event inventory + singleton anchors + three '
              'high-recall graph proposals + PRECONDITION-aware adjudication + '
              'posthoc stage recall diagnostics',
 'ok': True,
 'patch_version': 'unified4-v1.8'}


c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Controls

The paid run is enabled by default because this notebook is specifically for the requested parallel 5-document experiment.


In [2]:
RUN_PAID = True
RUN_ORDER = ["eventstoryline", "fincausal"]

MODEL_NAME = "openai/gpt-oss-20b"
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
REASONING_EFFORT = "minimal"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 180
VERBOSE = True

DOCUMENT_WORKERS = {
    "eventstoryline": 5,
    "fincausal": 5,
}

# 4 layer workers ONLY for EventStoryLine, as requested.
LAYER_WORKERS = {
    "eventstoryline": 4,
    "fincausal": 1,
}

EXPECTED_VERSIONS = {
    "eventstoryline": "v1.7",
    "fincausal": "unified-v1.3.1-selection-hotfix",
}

assert DOCUMENT_WORKERS["eventstoryline"] == 5
assert DOCUMENT_WORKERS["fincausal"] == 5
assert LAYER_WORKERS["eventstoryline"] == 4
assert LAYER_WORKERS["fincausal"] == 1

print("RUN_PAID:", RUN_PAID)
print("EventStoryLine: doc workers =", DOCUMENT_WORKERS["eventstoryline"],
      "| layer workers =", LAYER_WORKERS["eventstoryline"])
print("FinCausal     : doc workers =", DOCUMENT_WORKERS["fincausal"],
      "| layer workers =", LAYER_WORKERS["fincausal"])


RUN_PAID: True
EventStoryLine: doc workers = 5 | layer workers = 4
FinCausal     : doc workers = 5 | layer workers = 1


## Zero-cost preflight and exact fixed 5-document selection

EventStoryLine is resolved by the five fixed document IDs used previously.

FinCausal is resolved by **title**, not only `document_id`, because the first two fixed examples share the same document ID.


In [3]:
RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
ONTOLOGY_ROOT = expstate.discover_ontology_dir(RAGTREE_ROOT)

RAW_ONTOLOGY_FILES = expstate.locate_ontology_files(ONTOLOGY_ROOT)
DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)

ONTOLOGY_FILES = {
    "eventstoryline": RAW_ONTOLOGY_FILES["eventstoryline"],
    "fincausal": RAW_ONTOLOGY_FILES["fincausal"],
}

CONFIGS = {
    "eventstoryline": {
        "profile": EXPERIMENT_ROOT / "configs/eventstoryline_profile_native_ablation_v1_7.json",
        "guidance": EXPERIMENT_ROOT / "configs/guidance_eventstoryline_native_ablation_v1_7.json",
        "task": EXPERIMENT_ROOT / "configs/eventstoryline_task_guidance_v1_7.json",
        "catalog": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_aliases.json",
        "version": "v1.7",
    },
    "fincausal": {
        "profile": EXPERIMENT_ROOT / "configs/fincausal_profile_unified_v1_3.json",
        "guidance": EXPERIMENT_ROOT / "configs/fincausal_guidance_unified_v1_3.json",
        "task": EXPERIMENT_ROOT / "configs/fincausal_task_guidance_unified_v1_3.json",
        "catalog": EXPERIMENT_ROOT / "ontology/fincausal_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/fincausal_relation_aliases.json",
        "version": "unified-v1.3.1-selection-hotfix",
    },
}

for k, cfg in CONFIGS.items():
    assert cfg["version"] == EXPECTED_VERSIONS[k]
    for name, p in cfg.items():
        if name != "version":
            assert Path(p).exists(), (k, name, p)

dataset_rows = {
    k: expstate.read_jsonl(DATASET_FILES[k])
    for k in RUN_ORDER
}

# Read-only frozen smoke/version gate.
STATE_DIR = EXPERIMENT_ROOT / "state"
TEMPLATE_MANIFEST = STATE_DIR / "development_manifest_TEMPLATE_v1.json"
LIVE_MANIFEST = STATE_DIR / "development_manifest_v1.json"
dev_manifest = expstate.load_manifest(LIVE_MANIFEST, TEMPLATE_MANIFEST)

for k in RUN_ORDER:
    e = dev_manifest[k]
    assert e.get("one_doc_completed"), f"{k}: one-doc gate incomplete"
    assert e.get("smoke5_already_run"), f"{k}: smoke-5 gate incomplete"
    assert e.get("best_version") == EXPECTED_VERSIONS[k], (
        k, e.get("best_version"), EXPECTED_VERSIONS[k]
    )

FIXED_ESL_DOCUMENT_IDS = [
    "EventStoryLine - 1_10ecbplus",
    "EventStoryLine - 1_11ecbplus",
    "EventStoryLine - 1_12ecbplus",
    "EventStoryLine - 1_13ecbplus",
    "EventStoryLine - 1_14ecbplus",
]

FIXED_FINCAUSAL_TITLES = [
    "0001.00005.1",
    "0001.00005.2",
    "0001.00007",
    "0001.00009",
    "0001.00011",
]

def unique_lookup(rows, field, values):
    out = []
    for value in values:
        matches = [r for r in rows if str(r.get(field)) == str(value)]
        assert len(matches) == 1, (field, value, len(matches))
        out.append(matches[0])
    return out

fixed_rows = {
    "eventstoryline": unique_lookup(
        dataset_rows["eventstoryline"], "document_id", FIXED_ESL_DOCUMENT_IDS
    ),
    "fincausal": unique_lookup(
        dataset_rows["fincausal"], "title", FIXED_FINCAUSAL_TITLES
    ),
}

for k in RUN_ORDER:
    assert len(fixed_rows[k]) == 5
    keys = [expstate.record_key(k, r) for r in fixed_rows[k]]
    assert len(set(keys)) == 5, (k, keys)

    # Full anti-leak check on all five selected records.
    for r in fixed_rows[k]:
        clean = expstate.strip_gold(r)
        forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean)
        assert not forbidden, (k, r.get("document_id"), forbidden)

# FinCausal five must be the intended positive sample.
fc_contracts = [
    expstate.gold_contract_summary("fincausal", r)
    for r in fixed_rows["fincausal"]
]
for c in fc_contracts:
    assert c["gold_target_relation_count"] == 1, c
    assert c["gold_entity_count"] >= 2, c

print("Fixed EventStoryLine records:")
for r in fixed_rows["eventstoryline"]:
    print(" -", r.get("document_id"), "|", expstate.record_key("eventstoryline", r))

print("\nFixed FinCausal records:")
for r in fixed_rows["fincausal"]:
    print(" -", r.get("title"), "|", expstate.record_key("fincausal", r))

print("\nGold isolation: OK")
print("Frozen smoke/version gate: OK")
print("No API call has been made by this cell.")


AssertionError: ('document_id', 'EventStoryLine - 1_10ecbplus', 2)

## Separate parallel-test state

This experiment writes only under:

`examples/RAGTreeDatasets/runs/parallel5_eventstoryline_fincausal_v1`

It does **not** modify the development manifest. A completed document is skipped on notebook rerun.


In [ ]:
RUNS_ROOT = EXPERIMENT_ROOT / "runs" / "parallel5_eventstoryline_fincausal_v1"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

PROGRESS_PATH = RUNS_ROOT / "parallel5_progress.json"
SUMMARY_PATH = RUNS_ROOT / "parallel5_summary.json"
USAGE_WINDOW_PATH = RUNS_ROOT / "openrouter_usage_window.json"

progress_lock = threading.Lock()

def atomic_json(path, obj):
    expstate.atomic_write_json(Path(path), obj)

def load_json(path, default=None):
    p = Path(path)
    if not p.exists():
        return default
    return json.loads(p.read_text(encoding="utf-8"))

def fresh_progress():
    return {
        "schema_version": 1,
        "experiment": "parallel5_eventstoryline_fincausal_v1",
        "created_at_utc": utc_now(),
        "experiment_started_at_utc": None,
        "experiment_finished_at_utc": None,
        "model": MODEL_NAME,
        "dataset_order": list(RUN_ORDER),
        "datasets": {
            k: {
                "version": EXPECTED_VERSIONS[k],
                "document_workers": DOCUMENT_WORKERS[k],
                "layer_workers": LAYER_WORKERS[k],
                "record_keys": [expstate.record_key(k, r) for r in fixed_rows[k]],
                "completed_record_keys": [],
                "failures": {},
                "started_at_utc": None,
                "finished_at_utc": None,
            }
            for k in RUN_ORDER
        },
    }

progress = load_json(PROGRESS_PATH, None)
if progress is None:
    progress = fresh_progress()
    atomic_json(PROGRESS_PATH, progress)
else:
    assert progress["model"] == MODEL_NAME
    assert progress["dataset_order"] == RUN_ORDER
    for k in RUN_ORDER:
        assert progress["datasets"][k]["version"] == EXPECTED_VERSIONS[k]
        assert progress["datasets"][k]["document_workers"] == DOCUMENT_WORKERS[k]
        assert progress["datasets"][k]["layer_workers"] == LAYER_WORKERS[k]
        assert progress["datasets"][k]["record_keys"] == [
            expstate.record_key(k, r) for r in fixed_rows[k]
        ]

def write_usage_window():
    usage = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "experiment_started_at_utc": progress.get("experiment_started_at_utc"),
        "experiment_finished_at_utc": progress.get("experiment_finished_at_utc"),
        "datasets": {
            k: {
                "version": progress["datasets"][k]["version"],
                "document_workers": progress["datasets"][k]["document_workers"],
                "layer_workers": progress["datasets"][k]["layer_workers"],
                "started_at_utc": progress["datasets"][k].get("started_at_utc"),
                "finished_at_utc": progress["datasets"][k].get("finished_at_utc"),
            }
            for k in RUN_ORDER
        },
        "note": "UTC windows for later OpenRouter activity/token accounting.",
    }
    atomic_json(USAGE_WINDOW_PATH, usage)
    return usage

write_usage_window()

print("RUNS_ROOT:", RUNS_ROOT)
for k in RUN_ORDER:
    e = progress["datasets"][k]
    print(
        k,
        "completed=", len(e["completed_record_keys"]), "/5",
        "| doc_workers=", e["document_workers"],
        "| layer_workers=", e["layer_workers"],
    )


## Per-document native runner

Every worker receives a different document and a different run directory.

No shared development state is mutated from worker threads.


In [ ]:
def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

def run_one_record(dataset_key, gold_record):
    cfg = CONFIGS[dataset_key]
    rkey = expstate.record_key(dataset_key, gold_record)
    run_dir = RUNS_ROOT / dataset_key / "parallel5" / safe_dir_name(rkey)

    # Retry only the current record: clean its isolated directory first.
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    pre_gold_contract = expstate.gold_contract_summary(dataset_key, gold_record)

    clean_record = expstate.strip_gold(gold_record)
    forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean_record)
    assert not forbidden, (dataset_key, rkey, forbidden)

    input_path = run_dir / "pipeline_input_NO_GOLD.jsonl"
    expstate.write_jsonl(input_path, [clean_record])

    gold_path = run_dir / "POSTHOC_GOLD_AFTER_LAYER12.jsonl"
    assert not gold_path.exists()

    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    started = utc_now()
    t0 = time.perf_counter()

    if dataset_key == "eventstoryline":
        final_state = esl_v17.run_native_pipeline(
            project_root=PROJECT_ROOT,
            input_jsonl=input_path,
            ontology_path=ONTOLOGY_FILES[dataset_key],
            profile_path=cfg["profile"],
            guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"],
            relation_aliases_path=cfg["aliases"],
            run_dir=run_dir,
            model_name=MODEL_NAME,
            api_key=api_key,
            host=OPENROUTER_HOST,
            workers=LAYER_WORKERS[dataset_key],
            max_tokens=MAX_TOKENS,
            request_timeout=REQUEST_TIMEOUT,
            reasoning_effort=REASONING_EFFORT,
            verbose=VERBOSE,
            clean_run_dir=False,
        )

        # Gold only after native Layer 12 returns.
        expstate.write_jsonl(
            gold_path,
            [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
        )
        summary = esl_v17.analyze_run(
            run_dir=run_dir,
            gold_jsonl=gold_path,
            catalog_path=cfg["catalog"],
            aliases_path=cfg["aliases"],
        )
        result = {
            "dataset": dataset_key,
            "version": cfg["version"],
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "relation_metrics": (
                summary.get("projected_relation_evaluation")
                or summary.get("strict_relation_evaluation")
                or {}
            ),
            "endpoint_metrics": (
                summary.get("relation_endpoint_evaluation")
                or summary.get("event_entity_evaluation")
                or {}
            ),
            "candidate_pool": (
                summary.get("candidate_pool_coverage")
                or summary.get("candidate_pool")
                or {}
            ),
            "run_dir": str(run_dir),
            "pre_run_gold_contract": pre_gold_contract,
        }

    elif dataset_key == "fincausal":
        final_state = adapters.run_native_pipeline_record(
            dataset_key=dataset_key,
            project_root=PROJECT_ROOT,
            input_jsonl=input_path,
            ontology_path=ONTOLOGY_FILES[dataset_key],
            profile_path=cfg["profile"],
            guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"],
            relation_aliases_path=cfg["aliases"],
            run_dir=run_dir,
            model_name=MODEL_NAME,
            api_key=api_key,
            host=OPENROUTER_HOST,
            workers=LAYER_WORKERS[dataset_key],
            max_tokens=MAX_TOKENS,
            request_timeout=REQUEST_TIMEOUT,
            reasoning_effort=REASONING_EFFORT,
            verbose=VERBOSE,
            clean_run_dir=False,
        )

        # Gold only after native Layer 12 returns.
        expstate.write_jsonl(
            gold_path,
            [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
        )
        result = adapters.evaluate_state(dataset_key, final_state, gold_record)
        result.update({
            "dataset": dataset_key,
            "version": cfg["version"],
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "run_dir": str(run_dir),
            "pre_run_gold_contract": pre_gold_contract,
        })

        # Positive-gold evaluator-integrity guard.
        expected_gold = int(pre_gold_contract["gold_target_relation_count"])
        evaluated_gold = int((result.get("relation_metrics") or {}).get("gold", 0) or 0)
        if expected_gold > 0 and evaluated_gold == 0:
            raise RuntimeError(
                "FinCausal evaluation-integrity error AFTER Layer 12: "
                f"controller saw {expected_gold} gold CAUSE relation(s), evaluator saw 0."
            )
    else:
        raise ValueError(dataset_key)

    result["started_at_utc"] = started
    result["finished_at_utc"] = utc_now()
    result["elapsed_seconds"] = time.perf_counter() - t0
    result["document_workers"] = DOCUMENT_WORKERS[dataset_key]
    result["layer_workers"] = LAYER_WORKERS[dataset_key]
    result["gold_visible_to_pipeline"] = False

    adapters.write_json(run_dir / "posthoc_evaluation.json", result)
    return result

print("Parallel per-document runner defined. No API call has been made by this cell.")


## Aggregation helpers

Metric-key normalization handles the EventStoryLine evaluator (`true_positive`, `predicted`, etc.) and the FinCausal evaluator (`tp`, `pred`, etc.).


In [ ]:
def _metric_count(m, *names):
    if not isinstance(m, dict):
        return 0
    for name in names:
        if name in m and m[name] is not None:
            return int(m[name] or 0)
    return 0

def normalize_counts(m):
    tp = _metric_count(m, "tp", "true_positive")
    fp = _metric_count(m, "fp", "false_positive")
    fn = _metric_count(m, "fn", "false_negative")
    pred = _metric_count(m, "pred", "predicted")
    gold = _metric_count(m, "gold", "gold_unique")
    if pred == 0 and (tp + fp) > 0:
        pred = tp + fp
    if gold == 0 and (tp + fn) > 0:
        gold = tp + fn
    return {"pred": pred, "gold": gold, "tp": tp, "fp": fp, "fn": fn}

def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2*p*r/(p+r) if (p+r) else 0.0
    return p, r, f1

def result_path(dataset_key, rkey):
    return (
        RUNS_ROOT / dataset_key / "parallel5" /
        safe_dir_name(rkey) / "posthoc_evaluation.json"
    )

def load_results(dataset_key):
    completed = set(progress["datasets"][dataset_key]["completed_record_keys"])
    rows = []
    for r in fixed_rows[dataset_key]:
        rkey = expstate.record_key(dataset_key, r)
        if rkey not in completed:
            continue
        p = result_path(dataset_key, rkey)
        assert p.exists(), (dataset_key, rkey, p)
        rows.append(json.loads(p.read_text(encoding="utf-8")))
    return rows

def aggregate(dataset_key):
    rows = load_results(dataset_key)
    counts = [normalize_counts(r.get("relation_metrics") or {}) for r in rows]
    tp = sum(x["tp"] for x in counts)
    fp = sum(x["fp"] for x in counts)
    fn = sum(x["fn"] for x in counts)
    pred = sum(x["pred"] for x in counts)
    gold = sum(x["gold"] for x in counts)
    p, r, f1 = prf(tp, fp, fn)

    macro = (
        sum(float((x.get("relation_metrics") or {}).get("f1", 0.0) or 0.0) for x in rows)
        / len(rows)
        if rows else 0.0
    )

    return {
        "dataset": dataset_key,
        "version": EXPECTED_VERSIONS[dataset_key],
        "docs": len(rows),
        "pred": pred,
        "gold": gold,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": p,
        "recall": r,
        "micro_f1": f1,
        "mean_doc_f1": macro,
        "document_workers": DOCUMENT_WORKERS[dataset_key],
        "layer_workers": LAYER_WORKERS[dataset_key],
    }

def save_summary():
    summary = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "experiment_started_at_utc": progress.get("experiment_started_at_utc"),
        "experiment_finished_at_utc": progress.get("experiment_finished_at_utc"),
        "datasets": {k: aggregate(k) for k in RUN_ORDER},
    }
    atomic_json(SUMMARY_PATH, summary)
    write_usage_window()
    return summary

print("Aggregation helpers ready.")


## Execute parallel 5-document experiment

For each dataset, the five pending records are submitted to one `ThreadPoolExecutor`.

**EventStoryLine launches five documents concurrently with 4 native layer workers per document.**

After all five EventStoryLine futures finish, FinCausal launches its five documents concurrently with 1 native layer worker per document.


In [ ]:
if not RUN_PAID:
    print("RUN_PAID=False -> stopped before all API calls.")
else:
    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    with progress_lock:
        if progress.get("experiment_started_at_utc") is None:
            progress["experiment_started_at_utc"] = utc_now()
            atomic_json(PROGRESS_PATH, progress)
            write_usage_window()

    for dataset_key in RUN_ORDER:
        ds_state = progress["datasets"][dataset_key]

        # Strict dataset sequencing.
        if dataset_key == "fincausal":
            esl_done = len(progress["datasets"]["eventstoryline"]["completed_record_keys"])
            if esl_done != 5:
                raise RuntimeError(
                    f"FinCausal blocked: EventStoryLine has only {esl_done}/5 completed."
                )

        completed = set(ds_state.get("completed_record_keys") or [])
        pending = [
            r for r in fixed_rows[dataset_key]
            if expstate.record_key(dataset_key, r) not in completed
        ]

        if not pending:
            print(f"\nSKIP {dataset_key}: 5/5 already complete.")
            continue

        with progress_lock:
            if ds_state.get("started_at_utc") is None:
                ds_state["started_at_utc"] = utc_now()
                atomic_json(PROGRESS_PATH, progress)
                write_usage_window()

        print(
            f"\n=== {dataset_key.upper()} PARALLEL-5 ===\n"
            f"pending={len(pending)} | "
            f"DOCUMENT_WORKERS={DOCUMENT_WORKERS[dataset_key]} | "
            f"LAYER_WORKERS={LAYER_WORKERS[dataset_key]}"
        )

        futures = {}
        with ThreadPoolExecutor(
            max_workers=DOCUMENT_WORKERS[dataset_key],
            thread_name_prefix=f"neoolaf-{dataset_key}",
        ) as executor:
            for gold_record in pending:
                rkey = expstate.record_key(dataset_key, gold_record)
                fut = executor.submit(run_one_record, dataset_key, gold_record)
                futures[fut] = {
                    "record_key": rkey,
                    "document_id": gold_record.get("document_id"),
                    "title": gold_record.get("title"),
                }

            for fut in as_completed(futures):
                info = futures[fut]
                rkey = info["record_key"]

                try:
                    result = fut.result()

                    with progress_lock:
                        done = ds_state.setdefault("completed_record_keys", [])
                        if rkey not in done:
                            done.append(rkey)
                        ds_state.setdefault("failures", {}).pop(rkey, None)
                        atomic_json(PROGRESS_PATH, progress)
                        write_usage_window()

                    print(
                        f"\nDONE {dataset_key} | {rkey} | "
                        f"elapsed={result['elapsed_seconds']:.2f}s"
                    )
                    print(" relation:", result.get("relation_metrics"))
                    print(" endpoint:", result.get("endpoint_metrics"))

                except Exception as exc:
                    failure = {
                        **info,
                        "error_type": type(exc).__name__,
                        "error": str(exc),
                        "traceback": traceback.format_exc(),
                        "failed_at_utc": utc_now(),
                    }
                    with progress_lock:
                        ds_state.setdefault("failures", {})[rkey] = failure
                        atomic_json(PROGRESS_PATH, progress)
                        write_usage_window()

                    print(
                        f"\nFAILED {dataset_key} | {rkey} | "
                        f"{type(exc).__name__}: {exc}"
                    )

        with progress_lock:
            if len(ds_state.get("completed_record_keys") or []) == 5:
                ds_state["finished_at_utc"] = utc_now()
            atomic_json(PROGRESS_PATH, progress)
            write_usage_window()

        current = aggregate(dataset_key)
        print(f"\n{dataset_key} aggregate:")
        pprint(current)

        # Do not launch the next dataset after an incomplete parallel batch.
        if len(ds_state.get("completed_record_keys") or []) != 5:
            raise RuntimeError(
                f"{dataset_key}: only "
                f"{len(ds_state.get('completed_record_keys') or [])}/5 completed. "
                "Rerun the notebook; successful records will be skipped."
            )

    with progress_lock:
        if all(len(progress["datasets"][k]["completed_record_keys"]) == 5 for k in RUN_ORDER):
            progress["experiment_finished_at_utc"] = utc_now()
        atomic_json(PROGRESS_PATH, progress)
        write_usage_window()

    final_summary = save_summary()
    print("\nPARALLEL-5 EXPERIMENT COMPLETE")
    pprint(final_summary)
    print("\nProgress:", PROGRESS_PATH)
    print("Summary:", SUMMARY_PATH)
    print("OpenRouter UTC usage window:", USAGE_WINDOW_PATH)


## Final report — zero API calls

After execution, send me this executed notebook or the generated `parallel5_summary.json`.

The OpenRouter window file gives the exact UTC interval for this parallel experiment.


In [ ]:
summary = save_summary()
usage = write_usage_window()

print("PARALLEL-5 RESULTS")
print("==================")
for k in RUN_ORDER:
    a = summary["datasets"][k]
    print(
        f"\n{k} [{a['version']}] docs={a['docs']}/5 "
        f"| doc_workers={a['document_workers']} "
        f"| layer_workers={a['layer_workers']}"
    )
    print(
        f"P={a['precision']:.6f} "
        f"R={a['recall']:.6f} "
        f"micro-F1={a['micro_f1']:.6f} "
        f"mean-doc-F1={a['mean_doc_f1']:.6f} "
        f"TP={a['tp']} FP={a['fp']} FN={a['fn']}"
    )

print("\nOPENROUTER USAGE WINDOW")
pprint(usage)

print("\nGenerated files:")
print(" -", PROGRESS_PATH)
print(" -", SUMMARY_PATH)
print(" -", USAGE_WINDOW_PATH)
